In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


In [2]:
df = pd.read_csv('movies_metadata.csv')

In [3]:
df.head(2)

,adult,belongs_to_collection,budget,genres,homepage,id,imdb_id,original_language,original_title,overview,...,release_date,revenue,runtime,spoken_languages,status,tagline,title,video,vote_average,vote_count
0,False,"{'id': 10194, 'name': 'Toy Story Collection', ...",30000000,"[{'id': 16, 'name': 'Animation'}, {'id': 35, '...",http://toystory.disney.com/toy-story,862,tt0114709,en,Toy Story,"Led by Woody, Andy's toys live happily in his ...",...,1995-10-30,373554033.0,81.0,"[{'iso_639_1': 'en', 'name': 'English'}]",Released,NaN,Toy Story,False,7.7,5415.0
1,False,NaN,65000000,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",NaN,8844,tt0113497,en,Jumanji,When siblings Judy and Peter discover an encha...,...,1995-12-15,262797249.0,104.0,"[{'iso_639_1': 'en', 'name': 'English'}, {'iso...",Released,Roll the dice and unleash the excitement!,Jumanji,False,6.9,2413.0


In [4]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 45466 entries, 0 to 45465
Data columns (total 24 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   adult                  45466 non-null  str    
 1   belongs_to_collection  4494 non-null   str    
 2   budget                 45466 non-null  str    
 3   genres                 45466 non-null  str    
 4   homepage               7782 non-null   str    
 5   id                     45466 non-null  str    
 6   imdb_id                45449 non-null  str    
 7   original_language      45455 non-null  str    
 8   original_title         45466 non-null  str    
 9   overview               44512 non-null  str    
 10  popularity             45461 non-null  object 
 11  poster_path            45080 non-null  str    
 12  production_companies   45463 non-null  str    
 13  production_countries   45463 non-null  str    
 14  release_date           45379 non-null  str    
 15  revenue      

In [6]:
df.shape

(45466, 24)

In [7]:
# Let's check for null and duplicate values
df.isnull().sum()

adult                        0
belongs_to_collection    40972
budget                       0
genres                       0
homepage                 37684
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25054
title                        6
video                        6
vote_average                 6
vote_count                   6
dtype: int64

In [8]:
# A lot of null values in col -> belongs_to_collection,homepage,tagline
# We can't fill data like mean/mode in these cols due to such high amount of null values and being string data types.
# So we shouldn't use them for recommendation
# Although tagline maybe used (since tagline may help recommending similar movies like tagline of toystory 2 and toystory 3 may be similar)

In [9]:
# checking for duplicates and removing these

print("Duplicated rows -> ",df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
print("Duplicated rows now -> ",df.duplicated().sum())


Duplicated rows ->  13
Duplicated rows now ->  0


In [10]:
# Earlier rows - 45466
df.shape

(45453, 24)

In [11]:
df.columns

Index(['adult', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id',
       'imdb_id', 'original_language', 'original_title', 'overview',
       'popularity', 'poster_path', 'production_companies',
       'production_countries', 'release_date', 'revenue', 'runtime',
       'spoken_languages', 'status', 'tagline', 'title', 'video',
       'vote_average', 'vote_count'],
      dtype='str')

In [12]:
df = df[['title','overview','genres','tagline','vote_average','popularity']]

In [13]:
df.isnull().sum()

title               6
overview          954
genres              0
tagline         25045
vote_average        6
popularity          5
dtype: int64

In [14]:
# Since we can't predict title based on their overview and other columns, we have to drop rows with null titles
df = df.dropna(subset=['title'])

In [15]:
df.isnull().sum()

title               0
overview          954
genres              0
tagline         25039
vote_average        0
popularity          0
dtype: int64

In [16]:
# filling null overviews with empty spaces

df['overview'] = df['overview'].fillna('')

In [17]:
df.iloc[0].genres

"[{'id': 16, 'name': 'Animation'}, {'id': 35, 'name': 'Comedy'}, {'id': 10751, 'name': 'Family'}]"

In [18]:
# converting generes into list of name instead of json
from ast import literal_eval

df['genres'] = df['genres'].apply(lambda x: " ".join(i['name'] for i in literal_eval(x)))

In [19]:
df.iloc[0]

title                                                   Toy Story
overview        Led by Woody, Andy's toys live happily in his ...
genres                                    Animation Comedy Family
tagline                                                       NaN
vote_average                                                  7.7
popularity                                              21.946943
Name: 0, dtype: object

In [20]:
df['tagline'] = df['tagline'].fillna('')

In [21]:
df.isnull().sum()

title           0
overview        0
genres          0
tagline         0
vote_average    0
popularity      0
dtype: int64

In [22]:
# data is cleaned!!!

In [23]:
# Now creating a column tags consisting of overview title etc
df['tags'] = df['overview'] + " " + df['genres'] + ' ' + df['tagline']

In [24]:
df['tags'][0]

"Led by Woody, Andy's toys live happily in his room until Andy's birthday brings Buzz Lightyear onto the scene. Afraid of losing his place in Andy's heart, Woody plots against Buzz. But when circumstances separate Buzz and Woody from their owner, the duo eventually learns to put aside their differences. Animation Comedy Family "

In [25]:
import nltk
from nltk.stem import WordNetLemmatizer
import re
from nltk.corpus import stopwords


In [26]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [27]:
stop_words = set(stopwords.words('english'))
lemmatizer= WordNetLemmatizer()

In [28]:
def preprocess_text(text):
    text = str(text).lower()
    text =  re.sub(r'[^\w\s]', '', text) # Removing punctuations from the text

    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words] # lemmatizing

    return " ".join(words)

In [29]:
df['tags'] = df['tags'].apply(preprocess_text)

In [30]:
df['tags'][0]

'led woody andys toy live happily room andys birthday brings buzz lightyear onto scene afraid losing place andys heart woody plot buzz circumstance separate buzz woody owner duo eventually learns put aside difference animation comedy family'

In [31]:
df = df.reset_index(drop = True)

In [32]:
indices = pd.Series(df.index,index = df['title']).drop_duplicates()
indices

title
Toy Story                          0
Jumanji                            1
Grumpier Old Men                   2
Waiting to Exhale                  3
Father of the Bride Part II        4
                               ...  
Subdue                         45442
Century of Birthing            45443
Betrayal                       45444
Satan Triumphant               45445
Queerama                       45446
Length: 45447, dtype: int64

In [33]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [34]:
tfidf = TfidfVectorizer(stop_words='english',max_features=50000,ngram_range=(1,2)) 

In [35]:
tfidf_matrix = tfidf.fit_transform(df['tags'])

In [36]:
tfidf_matrix

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 1555624 stored elements and shape (45447, 50000)>

In [37]:
from sklearn.metrics.pairwise import cosine_similarity

In [38]:
def recommend(title, n = 10): # Title of the movie, no of movies to be recommended
    if title not in indices:
        return ['Movie not found']
    idx = indices[title]
    sim_score = cosine_similarity(tfidf_matrix[idx],tfidf_matrix).flatten() # Flatten so that there could be 3 4 5 or more dimensions, flattening it to 1D
    similar_idx = sim_score.argsort()[::-1][1:n+1]
    return df['title'].iloc[similar_idx]



In [47]:
recommend('Batman Begins')

35965                                    Batman: Bad Blood
19785              Batman: The Dark Knight Returns, Part 1
150                                         Batman Forever
21186    Batman Unmasked: The Psychology of the Dark Kn...
3094                          Batman: Mask of the Phantasm
21392                      Batman: Mystery of the Batwoman
18030                                     Batman: Year One
44961                             Batman Beyond: The Movie
15507                           Batman: Under the Red Hood
12853                                Batman: Gotham Knight
Name: title, dtype: str